# CS2 EXP-7 — NeoBERT-250M + LoRA


## 1. Dependencies


In [1]:
import sys
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121",
    "torch==2.5.1",
    "torchvision",
    "torchaudio",
    "transformers<4.49.0",  # Avoids the CVE check enforcing PyTorch 2.6
    "peft<0.14.0",
    "accelerate",
    "numpy<2.1.0",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "pyarrow",
    "joblib",
    "tqdm",
    "psutil",
    "einops",   # NeoBERT (chandar-lab/NeoBERT) remote modeling code dependency
], check=True)

# EXP-7 uses NeoBERT (chandar-lab/NeoBERT), which is shipped as `trust_remote_code`
# model code rather than a native `transformers` architecture. Its reference
# SwiGLU/attention implementation depends on `xformers`. xformers wheels are
# built against a SPECIFIC torch build, so we pin it explicitly to the release
# built for torch==2.5.1 (confirmed via its wheel metadata) rather than
# installing unpinned -- an unpinned `-U xformers` will pull the latest release,
# which requires a newer torch and will silently upgrade torch out from under
# the 2.5.1+cu121 pin above, which is what caused the earlier
# "xFormers was built for PyTorch 2.10.0+cu128" mismatch / flash-attention
# schema ImportError. We also use --no-deps so pip can't touch torch again to
# "satisfy" xformers. We force `use_unpadding=False` in case_study_2/models.py
# (see PDD sec. 5.2), so flash-attention is NOT required.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-deps",
    "xformers==0.0.28.post3",
], check=True)

print("Dependencies installed successfully for CUDA 12.1 driver!")

Dependencies installed successfully for CUDA 12.1 driver!


## 1.5 Settings


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "simo"

WORKSPACE_ROOT = Path.cwd()

REPO_ROOT = WORKSPACE_ROOT / "DiverseVul--IS-Project"
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

DATA_ROOT = WORKSPACE_ROOT / "IntelligentSystemProject" / "VulnerabilityDetectionData"

PROCESSED_DIR = DATA_ROOT / "processed"
MANIFEST_ROOT = DATA_ROOT / "manifests"
OUTPUT_ROOT = DATA_ROOT / "outputs"

SPLIT_ID = "cs1_project_holdout20_innercv_v1"

CODE_COLUMN = "normalized_code"
# CODE_COLUMN = "abstracted_code_v1"
CODE_COLUMN_TAG = "abstracted" if CODE_COLUMN == "abstracted_code_v1" else "normalized"

NORMALIZED_PARQUET = (
    PROCESSED_DIR
    / "rdiversevul_cs1_normalized_plus_abstracted_v2.parquet"
)

OUTER_MANIFEST_PATH = (
    MANIFEST_ROOT
    / SPLIT_ID
    / "outer_holdout"
    / "cs1_outer_project_holdout_manifest.parquet"
)

INNER_MANIFEST_PATH = (
    MANIFEST_ROOT
    / SPLIT_ID
    / "inner_cv"
    / "cs1_project_grouped_5fold_manifest.parquet"
)

EXP7_OUTPUT_DIR = (
    OUTPUT_ROOT
    / "case_study_2"
    / f"exp7_neobert_lora_v1_{CODE_COLUMN_TAG}"
)

EXP7_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR = WORKSPACE_ROOT / "IntelligentSystemProject" / "hf_cache"

HF_TOKEN_VALUE = ""
if HF_TOKEN_VALUE:
    os.environ["HF_TOKEN"] = HF_TOKEN_VALUE
else:
    os.environ.pop("HF_TOKEN", None)

RANK_GRID = (8, 16, 32)
EPOCHS = 4
SEARCH_EPOCHS = 2
SEARCH_N_SPLITS = 5

TRAIN_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 8

STORAGE_CAP_GB = 60

RUN_SMOKE_TEST = True
RUN_NESTED_OFFICIAL = True
RUN_CANONICAL_RETRAIN = True
RUN_HOLDOUT_EVAL = True

print("Settings loaded.")
print(f"Workspace: {WORKSPACE_ROOT}")
print(f"Repository: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Hugging Face cache: {HF_CACHE_DIR}")
print(f"Device: {DEVICE}")


## 2. Clone the repository


In [7]:
import urllib.request
import zipfile
from pathlib import Path

if not REPO_ROOT.exists():
    print(f"Downloading repository (branch: {REPO_BRANCH}) without git...")
    
    clean_url = REPO_URL.removesuffix(".git")
    zip_url = f"{clean_url}/archive/refs/heads/{REPO_BRANCH}.zip"
    zip_path = Path.cwd() / "repo_temp.zip"

   
    urllib.request.urlretrieve(zip_url, zip_path)

  
    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(Path.cwd())

  
    repo_name = clean_url.split("/")[-1]
    extracted_folder = Path.cwd() / f"{repo_name}-{REPO_BRANCH}"
    if extracted_folder.exists():
        extracted_folder.rename(REPO_ROOT)

 
    zip_path.unlink()
    print("Repository setup complete!")
else:
    print(f"Repository already exists at {REPO_ROOT}")


Repository already exists at /workspace/DiverseVul--IS-Project


## 3. Verify GPU, RAM, and storage budget


In [8]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
assert torch.cuda.is_available(), "CUDA is not available!"
assert torch.cuda.device_count() == 1, f"Expected 1 GPU, but PyTorch sees {torch.cuda.device_count()}"
DEVICE = "cuda:0"
print(f"Locked to single GPU: {torch.cuda.get_device_name(0)}")
print(f"Total Visible GPUs in PyTorch: {torch.cuda.device_count()}")

Locked to single GPU: NVIDIA A100-SXM4-80GB
Total Visible GPUs in PyTorch: 1


## 4. Data availability check


In [9]:
required_data_files = {
    "normalized parquet": NORMALIZED_PARQUET,
    "outer holdout manifest": OUTER_MANIFEST_PATH,
    "inner CV manifest": INNER_MANIFEST_PATH,
}

missing = {name: path for name, path in required_data_files.items() if not path.is_file()}

if missing:
    print("Missing required data files:")
    for name, path in missing.items():
        print(f"  - {name}: {path}")
    print(
        "\nThese files are produced by the Case Study 1 pipeline (normalization_v3.py + "
        "split_manifest.py) and were previously synced through Google Drive. On this machine, "
        "either:\n"
        "  1) Copy them from your previous Drive/Colab run into the paths above -- e.g. via the "
        "Jupyter file-browser upload, `scp`, or `rclone`/`gdown` from a terminal (you have root "
        "access here); or\n"
        "  2) Re-run the Case Study 1 notebook/pipeline against `data/raw/rdiversevul.json` to "
        "regenerate them locally.\n"
        f"Keep an eye on the {STORAGE_CAP_GB} GB storage cap while doing either."
    )
    raise FileNotFoundError("Required processed data/manifests are missing; see instructions above.")
else:
    for name, path in required_data_files.items():
        size_mb = path.stat().st_size / 1e6
        print(f"Found {name}: {path} ({size_mb:.1f} MB)")


Found normalized parquet: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_plus_abstracted_v1.parquet (210.7 MB)
Found outer holdout manifest: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/outer_holdout/cs1_outer_project_holdout_manifest.parquet (1.4 MB)
Found inner CV manifest: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/inner_cv/cs1_project_grouped_5fold_manifest.parquet (1.2 MB)


## 5. Write case_study_2 source files


In [10]:
(SRC_DIR / "case_study_2/__init__.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/__init__.py").write_text('')
print("Wrote", "case_study_2/__init__.py")


Wrote case_study_2/__init__.py


In [11]:
(SRC_DIR / "case_study_2/models.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/models.py").write_text('from __future__ import annotations\n\nimport os\nimport warnings\nfrom pathlib import Path\nfrom typing import Optional, Dict, Any, List\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom transformers import AutoConfig, AutoModel, AutoTokenizer\n\n\nDEFAULT_CODE_MODEL = "huggingface/CodeBERTa-small-v1"\nDEFAULT_CODE_TOKENIZER = "huggingface/CodeBERTa-small-v1"\n\n# EXP-7: NeoBERT-250M backbone (Chandar Research Lab). Plug-and-play replacement\n# for the CodeBERTa backbone above -- same hidden size (768), but a 4,096-token\n# RoPE/YaRE context window, SwiGLU activations, and Pre-RMSNorm. Ships as\n# `trust_remote_code` model code on the Hub rather than a native `transformers`\n# architecture, so it needs a couple of extra loading safeguards (see\n# `_neobert_loading_overrides` below).\nDEFAULT_NEOBERT_MODEL = "chandar-lab/NeoBERT"\nDEFAULT_NEOBERT_TOKENIZER = "chandar-lab/NeoBERT"\n\n# Model-name substrings that identify a NeoBERT-family checkpoint and therefore\n# require `trust_remote_code=True` plus the safeguards below. Kept as a simple\n# substring match (rather than a fixed set) so forks/finetunes of NeoBERT\n# (e.g. "chandar-lab/NeoBERT", "someuser/NeoBERT-finetuned-...") are still\n# picked up automatically.\n_NEOBERT_NAME_HINTS = ("neobert",)\n\n\ndef _is_neobert_model(model_name: str) -> bool:\n    name = (model_name or "").lower()\n    return any(hint in name for hint in _NEOBERT_NAME_HINTS)\n\n\ndef configure_huggingface_cache(hf_cache_dir: Optional[str] = None) -> None:\n    if hf_cache_dir:\n        hf_cache_dir = str(hf_cache_dir)\n        os.environ.setdefault("HF_HOME", hf_cache_dir)\n        os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(Path(hf_cache_dir) / "hub"))\n    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")\n    os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")\n    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")\n    os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "120")\n    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\n\ndef _dtype_from_policy(dtype_policy: str, device: str) -> Optional[torch.dtype]:\n    dtype_policy = (dtype_policy or "auto").lower()\n    device = str(device)\n    if dtype_policy == "float16":\n        return torch.float16 if device == "cuda" else torch.float32\n    if dtype_policy == "bfloat16":\n        return torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float32\n    if dtype_policy == "float32":\n        return torch.float32\n    if dtype_policy == "auto":\n        if device == "cuda" and torch.cuda.is_bf16_supported():\n            return torch.bfloat16\n        if device == "cuda":\n            return torch.float32\n        return torch.float32\n    raise ValueError(f"Unknown dtype_policy: {dtype_policy}")\n\n\ndef _apply_neobert_config_overrides(config: Any) -> Any:\n    """\n    Mandatory Track-B safeguard (PDD sec. 5.2, GitHub Issue #7 on\n    chandar-lab/NeoBERT -- "How is unpadding handled when unpacking?"):\n    sequence-packing / unpadding in the reference NeoBERT code can leak\n    attention across the pad boundary when the collator doesn\'t also emit\n    packed cu_seqlens, which is exactly our setup (we pad batches instead of\n    packing them). Forcing `use_unpadding=False` makes NeoBERT fall back to\n    strict padded multi-head attention with the HF attention mask, which is\n    the safe/correct path for this pipeline.\n\n    The exact attribute name has moved around across NeoBERT code revisions,\n    so we try the known aliases and only set whichever is actually present on\n    this revision\'s config, rather than hard-failing.\n    """\n    candidate_flags = ("use_unpadding", "unpad_inputs", "unpad", "pack_sequences")\n    matched = False\n    for flag in candidate_flags:\n        if hasattr(config, flag):\n            setattr(config, flag, False)\n            matched = True\n    if not matched:\n        warnings.warn(\n            "[models] Could not find a known unpadding flag on the NeoBERT config "\n            "(checked: %s). This revision of chandar-lab/NeoBERT may handle "\n            "padding differently -- double check attention-mask correctness "\n            "manually (see PDD sec. 5.2 / GitHub Issue #7)." % (candidate_flags,)\n        )\n    return config\n\n\ndef load_code_tokenizer(\n    tokenizer_name: str = DEFAULT_CODE_TOKENIZER,\n    hf_cache_dir: Optional[str] = None,\n    trust_remote_code: Optional[bool] = None,\n):\n    configure_huggingface_cache(hf_cache_dir)\n    if trust_remote_code is None:\n        trust_remote_code = _is_neobert_model(tokenizer_name)\n    return AutoTokenizer.from_pretrained(\n        tokenizer_name,\n        use_fast=True,\n        cache_dir=hf_cache_dir,\n        trust_remote_code=trust_remote_code,\n    )\n\n\ndef load_code_encoder(\n    model_name: str = DEFAULT_CODE_MODEL,\n    dtype_policy: str = "auto",\n    device: Optional[str] = None,\n    freeze: bool = True,\n    hf_cache_dir: Optional[str] = None,\n    trust_remote_code: Optional[bool] = None,\n) -> nn.Module:\n    device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    configure_huggingface_cache(hf_cache_dir)\n    dtype = _dtype_from_policy(dtype_policy, device)\n\n    is_neobert = _is_neobert_model(model_name)\n    if trust_remote_code is None:\n        trust_remote_code = is_neobert\n\n    kwargs: Dict[str, Any] = {"cache_dir": hf_cache_dir, "trust_remote_code": trust_remote_code}\n    if dtype is not None:\n        kwargs["torch_dtype"] = dtype\n\n    if is_neobert:\n        # Load + patch the config explicitly (rather than relying on\n        # AutoModel.from_pretrained\'s implicit config loading) so the\n        # unpadding override in _apply_neobert_config_overrides is guaranteed\n        # to be in effect before the backbone is instantiated.\n        config = AutoConfig.from_pretrained(\n            model_name, cache_dir=hf_cache_dir, trust_remote_code=trust_remote_code\n        )\n        config = _apply_neobert_config_overrides(config)\n        kwargs["config"] = config\n\n    model = AutoModel.from_pretrained(model_name, **kwargs)\n    model.to(device)\n\n    if freeze:\n        for param in model.parameters():\n            param.requires_grad = False\n        model.eval()\n\n    return model\n\n\ndef mean_pool_last_hidden(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)\n    summed = (last_hidden_state * mask).sum(dim=1)\n    denom = mask.sum(dim=1).clamp(min=1.0)\n    return summed / denom\n\n\ndef cls_pool_last_hidden(last_hidden_state: torch.Tensor) -> torch.Tensor:\n    return last_hidden_state[:, 0, :]\n\n\nclass CodeSequenceClassifier(nn.Module):\n    def __init__(\n        self,\n        model_name: str = DEFAULT_CODE_MODEL,\n        num_labels: int = 1,\n        freeze_backbone: bool = False,\n        pooling: str = "mean",\n        dtype_policy: str = "auto",\n        hf_cache_dir: Optional[str] = None,\n        trust_remote_code: Optional[bool] = None,\n        enforce_fp32_head: Optional[bool] = None,\n    ) -> None:\n        super().__init__()\n        device = "cuda" if torch.cuda.is_available() else "cpu"\n        self.backbone = load_code_encoder(\n            model_name=model_name,\n            dtype_policy=dtype_policy,\n            device=device,\n            freeze=freeze_backbone,\n            hf_cache_dir=hf_cache_dir,\n            trust_remote_code=trust_remote_code,\n        )\n        hidden_size = int(self.backbone.config.hidden_size)\n        self.classification_head = nn.Linear(hidden_size, num_labels)\n        self.pooling = pooling\n\n        # Mandatory Track-B safeguard (PDD sec. 5.2, GitHub Issue #11 on\n        # chandar-lab/NeoBERT -- NaN training bug): pooling + the\n        # classification head are forced to run in explicit float32,\n        # regardless of the ambient autocast dtype, so a bf16/fp16 NaN/Inf\n        # produced upstream in NeoBERT\'s attention stack doesn\'t get baked\n        # into the (trainable) head via a half-precision matmul. This is a\n        # wrapper-level mitigation -- it doesn\'t patch NeoBERT\'s own remote\n        # code, it just keeps *our* downstream math numerically safe.\n        if enforce_fp32_head is None:\n            enforce_fp32_head = _is_neobert_model(model_name)\n        self.enforce_fp32_head = enforce_fp32_head\n\n    @property\n    def config(self):\n        """Expose the underlying backbone config to pyreft/peft."""\n        return self.backbone.config\n\n    @property\n    def device(self) -> torch.device:\n        """Expose the device where parameters reside for pyreft."""\n        return next(self.parameters()).device\n\n    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, **kwargs: Any) -> torch.Tensor:\n        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, **kwargs)\n        hidden = outputs.last_hidden_state\n\n        if self.enforce_fp32_head:\n            hidden = hidden.float()\n            attention_mask_for_pool = attention_mask.float()\n        else:\n            attention_mask_for_pool = attention_mask\n\n        if self.pooling == "cls":\n            pooled = cls_pool_last_hidden(hidden)\n        else:\n            pooled = mean_pool_last_hidden(hidden, attention_mask_for_pool)\n\n        if self.enforce_fp32_head:\n            # Disable autocast for the head matmul so it isn\'t silently\n            # downcast back to bf16/fp16 by the enclosing `torch.amp.autocast`\n            # context in the training loop.\n            with torch.autocast(device_type=pooled.device.type, enabled=False):\n                logits = self.classification_head(pooled.float())\n        else:\n            logits = self.classification_head(pooled)\n\n        if logits.ndim > 1 and logits.size(-1) == 1:\n            return logits.squeeze(-1)\n        return logits\n\n\ndef count_trainable_parameters(model: nn.Module) -> Dict[str, int]:\n    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    total = sum(p.numel() for p in model.parameters())\n    return {\n        "trainable_parameters": int(trainable),\n        "total_parameters": int(total),\n        "trainable_percent": float(100.0 * trainable / max(total, 1)),\n    }\n\n\ndef infer_lora_target_modules(model: nn.Module) -> List[str]:\n    module_names = [name for name, _ in model.named_modules()]\n    candidate_sets = [\n        ["qkv"],\n        ["q_proj", "v_proj"],\n        ["query", "value"],\n        ["in_proj"],\n    ]\n    for candidates in candidate_sets:\n        if all(any(name.endswith(candidate) or f".{candidate}" in name for name in module_names) for candidate in candidates):\n            return candidates\n    return ["query", "value"]\n\n\ndef create_lora_sequence_classifier(\n    model_name: str = DEFAULT_CODE_MODEL,\n    rank: int = 8,\n    lora_alpha: int = 16,\n    lora_dropout: float = 0.05,\n    pooling: str = "mean",\n    dtype_policy: str = "auto",\n    hf_cache_dir: Optional[str] = None,\n    trust_remote_code: Optional[bool] = None,\n):\n    from peft import LoraConfig, get_peft_model\n\n    base = CodeSequenceClassifier(\n        model_name=model_name,\n        freeze_backbone=False,\n        pooling=pooling,\n        dtype_policy=dtype_policy,\n        hf_cache_dir=hf_cache_dir,\n        trust_remote_code=trust_remote_code,\n    )\n    target_modules = infer_lora_target_modules(base)\n    config = LoraConfig(\n        r=rank,\n        lora_alpha=lora_alpha,\n        target_modules=target_modules,\n        lora_dropout=lora_dropout,\n        bias="none",\n        task_type="FEATURE_EXTRACTION",\n        modules_to_save=["classification_head"],\n    )\n    return get_peft_model(base, config)\n\n\ndef get_lora_model(\n    model_name: str = DEFAULT_CODE_MODEL,\n    rank: int = 8,\n    lora_alpha: int = 16,\n    pooling: str = "mean",\n    trust_remote_code: Optional[bool] = None,\n):\n    return create_lora_sequence_classifier(\n        model_name=model_name,\n        rank=rank,\n        lora_alpha=lora_alpha,\n        pooling=pooling,\n        trust_remote_code=trust_remote_code,\n    )\n\n\n# =====================================================================\n# EXP-5: HEFT (Hierarchical Efficient Fine-Tuning: LoRA -> freeze -> ReFT)\n# =====================================================================\n#\n# HEFT is a two-phase procedure:\n#   Phase 1 (LoRA): train a standard LoRA-adapted sequence classifier\n#                    (use get_lora_model() below, already defined above).\n#   Phase 2 (ReFT):  freeze *everything* learned in Phase 1 (LoRA adapters,\n#                    classification head, backbone) and train a LoReFT\n#                    intervention on top of the frozen, LoRA-adapted backbone\n#                    (use attach_reft_to_lora_model() below).\n#\n# These two builders are meant to be driven by the two-phase training loop in\n# exp5_heft.py -- they only construct models, they don\'t train anything.\n\ndef freeze_lora_parameters(model: nn.Module) -> None:\n    """Freezes LoRA adapter parameters learned in Phase 1."""\n    for name, param in model.named_parameters():\n        if "lora_" in name:\n            param.requires_grad = False\n\n\ndef reft_component_path(layer_target: int) -> str:\n    """\n    Dotted/bracket component path pyreft needs to locate the target layer\'s\n    output *inside a peft-wrapped model*.\n\n    peft.get_peft_model() re-nests the original module tree under\n    "base_model.model.*" rather than preserving the original top-level\n    attribute names. pyreft/pyvene resolve `component` strings via\n    nn.Module.get_submodule(), which walks the *real* module registry (not\n    Python attribute-forwarding), so a path like\n    "backbone.encoder.layer[i].output" -- valid on the bare, unwrapped model --\n    does not exist once the model has been wrapped with LoRA, and pyreft will\n    fail to find it. It must be prefixed with "base_model.model." to match the\n    wrapped model\'s actual module tree. (This mirrors the pattern pyreft\'s own\n    docs use for peft-wrapped causal LMs: "base_model.model.model.layers[i]...".)\n    """\n    return f"base_model.model.backbone.encoder.layer[{layer_target}].output"\n\n\ndef attach_reft_to_lora_model(\n    lora_model: nn.Module,\n    reft_rank: int = 4,\n    layer_target: int = 4,\n    freeze_previous_phase: bool = True,\n):\n    """\n    HEFT Phase 2. Takes an already Phase-1-trained LoRA model (as returned by\n    get_lora_model / create_lora_sequence_classifier) and attaches a LoReFT\n    intervention on top of it.\n\n    Freezing behaviour:\n      - `freeze_previous_phase=True` explicitly freezes the LoRA adapter\n        parameters first (belt-and-braces).\n      - pyreft.get_reft_model() *also* freezes every remaining parameter of\n        the wrapped model by design -- that\'s the whole point of ReFT: adapt\n        frozen representations via a small intervention instead of updating\n        weights. So after this call, the classification head and backbone end\n        up frozen too; only the newly added LoReFT intervention parameters\n        are trainable. That matches "train LoRA, freeze it, apply ReFT".\n    """\n    import pyreft\n\n    if freeze_previous_phase:\n        freeze_lora_parameters(lora_model)\n\n    hidden_size = int(lora_model.base_model.model.backbone.config.hidden_size)\n    component_path = reft_component_path(layer_target)\n\n    # NOTE: pyreft.ReftConfig expects `representations` as plain dict(s), NOT a\n    # pyreft.RepresentationConfig object -- no such class exists in pyreft\'s\n    # public API. Every real example in pyreft\'s own README/docs builds it this way.\n    reft_config = pyreft.ReftConfig(\n        representations=[\n            {\n                "layer": layer_target,\n                "component": component_path,\n                "low_rank_dimension": reft_rank,\n                "intervention": pyreft.LoreftIntervention(\n                    embed_dim=hidden_size,\n                    low_rank_dimension=reft_rank,\n                ),\n            }\n        ]\n    )\n\n    # set_device=False prevents PyReft from probing custom module properties during init\n    heft_model = pyreft.get_reft_model(lora_model, reft_config, set_device=False)\n\n    return heft_model\n')
print("Wrote", "case_study_2/models.py")


Wrote case_study_2/models.py


In [12]:
(SRC_DIR / "case_study_2/data_loader.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/data_loader.py").write_text('from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Optional, Dict, Any, List, Tuple\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\n\n\nEMPTY_CODE_SENTINEL = "EMPTY_CODE_SAMPLE"\n\n\nclass CodeTextDataset(Dataset):\n    def __init__(\n        self,\n        dataframe: pd.DataFrame,\n        code_column: str = "normalized_code",\n        label_column: str = "label",\n        source_id_column: str = "source_row_id",\n        project_column: str = "project",\n    ) -> None:\n        self.df = dataframe.copy().reset_index(drop=True)\n        self.code_column = code_column\n        self.label_column = label_column\n        self.source_id_column = source_id_column\n        self.project_column = project_column\n\n        self.df[self.code_column] = self.df[self.code_column].fillna("").astype(str)\n        empty_mask = self.df[self.code_column].str.strip().eq("")\n        if empty_mask.any():\n            self.df.loc[empty_mask, self.code_column] = EMPTY_CODE_SENTINEL\n\n    def __len__(self) -> int:\n        return int(len(self.df))\n\n    def __getitem__(self, idx: int) -> Dict[str, Any]:\n        row = self.df.iloc[idx]\n        return {\n            "code": str(row[self.code_column]),\n            "label": int(row[self.label_column]),\n            "source_row_id": int(row[self.source_id_column]),\n            "project": str(row[self.project_column]),\n        }\n\n\n@dataclass\nclass TransformerBatchCollator:\n    tokenizer: Any\n    max_length: int = 512\n    pad_to_multiple_of: Optional[int] = 8\n\n    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:\n        texts = [feature["code"] for feature in features]\n        enc = self.tokenizer(\n            texts,\n            truncation=True,\n            max_length=self.max_length,\n            padding=True,\n            pad_to_multiple_of=self.pad_to_multiple_of,\n            return_tensors="pt",\n        )\n\n        labels = torch.tensor([feature["label"] for feature in features], dtype=torch.float32)\n        source_row_ids = torch.tensor([feature["source_row_id"] for feature in features], dtype=torch.long)\n        projects = [feature["project"] for feature in features]\n\n        enc["labels"] = labels\n        enc["label"] = labels\n        enc["source_row_id"] = source_row_ids\n        enc["project"] = projects\n        return enc\n\n\ndef create_dataloader(\n    dataframe: pd.DataFrame,\n    tokenizer: Any,\n    batch_size: int = 16,\n    max_length: int = 512,\n    shuffle: bool = False,\n    code_column: str = "normalized_code",\n    label_column: str = "label",\n    source_id_column: str = "source_row_id",\n    project_column: str = "project",\n    num_workers: int = 0,\n) -> DataLoader:\n    dataset = CodeTextDataset(\n        dataframe=dataframe,\n        code_column=code_column,\n        label_column=label_column,\n        source_id_column=source_id_column,\n        project_column=project_column,\n    )\n    collator = TransformerBatchCollator(\n        tokenizer=tokenizer,\n        max_length=max_length,\n        pad_to_multiple_of=8 if torch.cuda.is_available() else None,\n    )\n    return DataLoader(\n        dataset,\n        batch_size=batch_size,\n        shuffle=shuffle,\n        drop_last=False,\n        num_workers=num_workers,\n        pin_memory=torch.cuda.is_available(),\n        collate_fn=collator,\n    )\n\n\ndef get_pos_weight(dataframe: pd.DataFrame, label_column: str = "label") -> torch.Tensor:\n    y = dataframe[label_column].astype(int).values\n    neg = int((y == 0).sum())\n    pos = int((y == 1).sum())\n    if pos == 0:\n        return torch.tensor([1.0], dtype=torch.float32)\n    return torch.tensor([neg / pos], dtype=torch.float32)\n\n\ndef get_class_weights(dataframe: pd.DataFrame, label_column: str = "label") -> torch.Tensor:\n    return get_pos_weight(dataframe, label_column=label_column)\n\n\ndef sample_with_optional_positive_fraction(\n    frame: pd.DataFrame,\n    n_rows: int,\n    label_column: str = "label",\n    positive_fraction: Optional[float] = None,\n    random_state: int = 42,\n) -> pd.DataFrame:\n    if n_rows is None or n_rows <= 0 or len(frame) <= n_rows:\n        return frame.copy().reset_index(drop=True)\n\n    rng = np.random.default_rng(random_state)\n\n    if positive_fraction is None:\n        indices = rng.choice(frame.index.to_numpy(), size=n_rows, replace=False)\n        return frame.loc[indices].copy().reset_index(drop=True)\n\n    positives = frame[frame[label_column].astype(int) == 1]\n    negatives = frame[frame[label_column].astype(int) == 0]\n\n    n_pos = min(len(positives), max(1, int(round(n_rows * positive_fraction))))\n    n_neg = min(len(negatives), n_rows - n_pos)\n\n    pos_idx = rng.choice(positives.index.to_numpy(), size=n_pos, replace=False) if n_pos else []\n    neg_idx = rng.choice(negatives.index.to_numpy(), size=n_neg, replace=False) if n_neg else []\n    idx = np.concatenate([pos_idx, neg_idx])\n    rng.shuffle(idx)\n\n    return frame.loc[idx].copy().reset_index(drop=True)\n\n\ndef make_project_disjoint_threshold_split(\n    train_frame: pd.DataFrame,\n    threshold_fraction: float = 0.20,\n    project_column: str = "project",\n    label_column: str = "label",\n    random_state: int = 42,\n) -> Tuple[pd.DataFrame, pd.DataFrame]:\n    projects = train_frame[[project_column, label_column]].groupby(project_column)[label_column].agg(["count", "sum"])\n    project_names = projects.index.to_numpy()\n\n    rng = np.random.default_rng(random_state)\n    shuffled = project_names.copy()\n    rng.shuffle(shuffled)\n\n    target_rows = int(round(len(train_frame) * threshold_fraction))\n    selected = []\n    count = 0\n\n    for project in shuffled:\n        selected.append(project)\n        count += int(projects.loc[project, "count"])\n        if count >= target_rows:\n            break\n\n    selected = set(selected)\n    threshold_mask = train_frame[project_column].isin(selected)\n    threshold_frame = train_frame[threshold_mask].copy().reset_index(drop=True)\n    fit_frame = train_frame[~threshold_mask].copy().reset_index(drop=True)\n\n    if (\n        fit_frame[label_column].sum() == 0\n        or threshold_frame[label_column].sum() == 0\n        or len(fit_frame) == 0\n        or len(threshold_frame) == 0\n    ):\n        shuffled_rows = train_frame.sample(frac=1.0, random_state=random_state).reset_index(drop=True)\n        cut = max(1, int(round(len(shuffled_rows) * (1.0 - threshold_fraction))))\n        fit_frame = shuffled_rows.iloc[:cut].copy().reset_index(drop=True)\n        threshold_frame = shuffled_rows.iloc[cut:].copy().reset_index(drop=True)\n\n    return fit_frame, threshold_frame\n')
print("Wrote", "case_study_2/data_loader.py")


Wrote case_study_2/data_loader.py


In [13]:
(SRC_DIR / "case_study_2/exp7/__init__.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/exp7/__init__.py").write_text('')
print("Wrote", "case_study_2/exp7/__init__.py")


Wrote case_study_2/exp7/__init__.py


In [14]:
(SRC_DIR / "case_study_2/exp7/exp7_lora.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/exp7/exp7_lora.py").write_text('from __future__ import annotations\n\nimport copy\nimport gc\nimport time\nimport torch\nimport torch.nn as nn\nfrom torch.optim import AdamW\nfrom sklearn.metrics import average_precision_score\nimport numpy as np\n\nfrom case_study_2.data_loader import create_dataloader, get_class_weights\nfrom case_study_2.models import get_lora_model, count_trainable_parameters, DEFAULT_NEOBERT_MODEL\n\n\ndef score_model(model, df, tokenizer, device, code_column="normalized_code",\n                 max_length=512, batch_size=32, num_workers=2):\n    """\n    Public scoring helper: run a trained model over an arbitrary dataframe\n    and return sigmoid scores as a numpy array, in the dataframe\'s row order.\n\n    Used by run_exp7_canonical_retrain for the FINAL, no-gradient pass over\n    the frozen outer holdout, kept separate from the early-stopping\n    validation loop (which must never see the holdout -- see handoff notes).\n    """\n    loader = create_dataloader(\n        df, tokenizer, batch_size=batch_size, max_length=max_length,\n        shuffle=False, num_workers=num_workers, code_column=code_column,\n    )\n    _, scores = _score_val_loader(model, loader, device)\n    return scores\n\n\ndef _score_val_loader(model, val_loader, device):\n    """Run inference over val_loader and return (y_true, y_score) numpy arrays."""\n    model.eval()\n    all_scores, all_labels = [], []\n    with torch.no_grad():\n        for batch in val_loader:\n            input_ids = batch["input_ids"].to(device, non_blocking=True)\n            attention_mask = batch["attention_mask"].to(device, non_blocking=True)\n            labels = batch["label"]\n            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):\n                logits = model(input_ids, attention_mask)\n                scores = torch.sigmoid(logits)\n            all_scores.extend(scores.float().cpu().numpy())\n            all_labels.extend(labels.numpy())\n    return np.array(all_labels), np.array(all_scores)\n\n\ndef train_lora_model(\n    train_df,\n    val_df,\n    tokenizer,\n    rank,\n    epochs=3,\n    batch_size=16,\n    grad_accum_steps=2,\n    eval_batch_size=32,\n    num_workers=2,\n    device="cuda",\n    hf_cache_dir=None,\n    code_column="normalized_code",\n    max_length=512,\n    verbose=True,\n    log_every_steps=500,\n    log_prefix="",\n    # --- FIX (undertraining diagnosis, see handoff notes) -----------------\n    # EXP-7 (NeoBERT, 28 layers) was inheriting the same fixed epoch budget\n    # tuned for EXP-4 (CodeBERTa, 6 layers). The training loss was still\n    # falling steadily at the last epoch (1.14 -> 1.02 -> 0.94 -> 0.83 over\n    # 4 epochs), i.e. training was stopped before convergence, not because\n    # it had plateaued. Rather than guess a new fixed epoch count, this adds\n    # validation-PR-AUC-based early stopping: keep training up to\n    # `max_epochs`, but stop once val PR-AUC hasn\'t improved for `patience`\n    # consecutive epochs, and return the BEST checkpoint seen (not the last).\n    # `epochs` is kept as the effective ceiling for backward compatibility\n    # with existing call sites (nested search / canonical retrain) that\n    # already pass `epochs=config.search_epochs` / `epochs=config.epochs`.\n    early_stopping=True,\n    patience=2,\n    min_epochs=2,\n):\n    train_loader = create_dataloader(\n        train_df, tokenizer, batch_size=batch_size, max_length=max_length,\n        shuffle=True, num_workers=num_workers, code_column=code_column,\n    )\n    val_loader = create_dataloader(\n        val_df, tokenizer, batch_size=eval_batch_size, max_length=max_length,\n        shuffle=False, num_workers=num_workers, code_column=code_column,\n    )\n\n    # EXP-7: identical LoRA recipe to EXP-4, only the backbone changes\n    # (CodeBERTa -> NeoBERT-250M). get_lora_model() auto-detects the NeoBERT\n    # name and applies trust_remote_code=True plus the unpadding/fp32-head\n    # safeguards defined in case_study_2/models.py.\n    model = get_lora_model(model_name=DEFAULT_NEOBERT_MODEL, rank=rank, lora_alpha=16).to(device)\n\n    is_cuda = (device == "cuda") or (hasattr(device, "type") and device.type == "cuda")\n\n    total_steps_per_epoch = -(-len(train_df) // batch_size)\n\n    if verbose:\n        stats = count_trainable_parameters(model)\n        print(\n            f"{log_prefix}[lora] rank={rank} | train_rows={len(train_df)} | val_rows={len(val_df)} | "\n            f"batch_size={batch_size} | grad_accum={grad_accum_steps} | steps/epoch={total_steps_per_epoch} | "\n            f"trainable={stats[\'trainable_parameters\']:,} ({stats[\'trainable_percent\']:.3f}%) | "\n            f"total={stats[\'total_parameters\']:,}"\n        )\n        if is_cuda:\n            print(f"{log_prefix}[lora] VRAM after model load: {torch.cuda.memory_allocated()/1e9:.2f} GB")\n\n    pos_weight = get_class_weights(train_df).to(device)\n    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)\n    optimizer = AdamW(model.parameters(), lr=2e-4)\n\n    t0 = time.time()\n\n    best_val_prauc = -1.0\n    best_epoch = -1\n    best_state_dict = None\n    best_scores = None\n    epochs_without_improvement = 0\n\n    for epoch in range(epochs):\n        model.train()\n        epoch_loss = 0.0\n        n_steps = 0\n        epoch_t0 = time.time()\n        optimizer.zero_grad()\n\n        for step, batch in enumerate(train_loader):\n            input_ids = batch["input_ids"].to(device, non_blocking=True)\n            attention_mask = batch["attention_mask"].to(device, non_blocking=True)\n            labels = batch["label"].to(device, non_blocking=True)\n\n            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):\n                logits = model(input_ids, attention_mask)\n                loss = criterion(logits, labels) / grad_accum_steps\n\n            loss.backward()\n            epoch_loss += loss.item() * grad_accum_steps\n            n_steps += 1\n\n            if (step + 1) % grad_accum_steps == 0:\n                optimizer.step()\n                optimizer.zero_grad()\n\n            if verbose and log_every_steps and (step + 1) % log_every_steps == 0:\n                elapsed_min = (time.time() - epoch_t0) / 60\n                steps_left = total_steps_per_epoch - (step + 1)\n                rate = (step + 1) / max(elapsed_min, 1e-6)\n                eta_min = steps_left / max(rate, 1e-6)\n                print(\n                    f"{log_prefix}[lora] epoch {epoch+1}/{epochs} step {step+1}/{total_steps_per_epoch} | "\n                    f"avg_loss_so_far={epoch_loss/max(n_steps,1):.4f} | "\n                    f"elapsed={elapsed_min:.1f} min | ETA epoch ~{eta_min:.1f} min"\n                )\n\n        optimizer.step()\n        optimizer.zero_grad()\n\n        # --- FIX: evaluate on val_df every epoch, track the best checkpoint ---\n        val_labels, val_scores_epoch = _score_val_loader(model, val_loader, device)\n        val_prauc = float(average_precision_score(val_labels, val_scores_epoch))\n        improved = val_prauc > best_val_prauc\n\n        if verbose:\n            elapsed_min = (time.time() - t0) / 60\n            epoch_min = (time.time() - epoch_t0) / 60\n            peak_vram = torch.cuda.max_memory_allocated() / 1e9 if is_cuda else 0.0\n            flag = " <- best so far" if improved else ""\n            print(\n                f"{log_prefix}[lora] epoch {epoch+1}/{epochs} done | avg_loss={epoch_loss/max(n_steps,1):.4f} "\n                f"| val_pr_auc={val_prauc:.4f}{flag} "\n                f"| epoch_time={epoch_min:.1f} min | total_elapsed={elapsed_min:.1f} min | peak_VRAM={peak_vram:.2f} GB"\n            )\n\n        if improved:\n            best_val_prauc = val_prauc\n            best_epoch = epoch + 1\n            best_scores = val_scores_epoch\n            epochs_without_improvement = 0\n            if early_stopping:\n                # Keep weights on CPU to avoid holding two full copies on GPU.\n                best_state_dict = copy.deepcopy(model.state_dict())\n                best_state_dict = {k: v.cpu() for k, v in best_state_dict.items()}\n        else:\n            epochs_without_improvement += 1\n\n        if early_stopping and (epoch + 1) >= min_epochs and epochs_without_improvement >= patience:\n            if verbose:\n                print(\n                    f"{log_prefix}[lora] early stopping at epoch {epoch+1}/{epochs} "\n                    f"(no val PR-AUC improvement for {patience} epochs; best was epoch {best_epoch}, "\n                    f"val_pr_auc={best_val_prauc:.4f})"\n                )\n            break\n\n    if verbose:\n        print(f"{log_prefix}[lora] training done ({epoch+1} epoch(s) run).")\n\n    if early_stopping and best_state_dict is not None:\n        # Restore the best-epoch weights before returning, so the caller\n        # (nested search / canonical retrain / holdout eval) always gets the\n        # checkpoint with the highest validation PR-AUC, not just the last one.\n        model.load_state_dict({k: v.to(device) for k, v in best_state_dict.items()})\n        all_scores = best_scores\n        if verbose:\n            print(f"{log_prefix}[lora] restored best checkpoint from epoch {best_epoch} (val_pr_auc={best_val_prauc:.4f}).")\n    else:\n        # early_stopping=False, or somehow no epoch ever improved (shouldn\'t\n        # happen since epoch 1 always sets best_*): fall back to last epoch\'s\n        # scores, matching the old (pre-fix) behaviour.\n        all_scores = best_scores if best_scores is not None else _score_val_loader(model, val_loader, device)[1]\n\n    del train_loader, val_loader, criterion, optimizer\n    if is_cuda:\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats()\n\n    return np.array(all_scores), model\n\n\ndef train_lora_model_safe(*args, max_retries=2, **kwargs):\n    batch_size = kwargs.pop("batch_size", 16)\n    grad_accum_steps = kwargs.pop("grad_accum_steps", 2)\n\n    attempt = 0\n    while True:\n        try:\n            return train_lora_model(\n                *args, batch_size=batch_size, grad_accum_steps=grad_accum_steps, **kwargs\n            )\n        except torch.cuda.OutOfMemoryError:\n            attempt += 1\n            gc.collect()\n            torch.cuda.empty_cache()\n            if attempt > max_retries or batch_size <= 2:\n                raise\n            new_batch_size = max(2, batch_size // 2)\n            new_grad_accum_steps = grad_accum_steps * max(1, batch_size // new_batch_size)\n            print(\n                f"[lora] CUDA OOM at batch_size={batch_size}; retrying "\n                f"(attempt {attempt}/{max_retries}) with batch_size={new_batch_size}, "\n                f"grad_accum_steps={new_grad_accum_steps} (effective batch size unchanged)."\n            )\n            batch_size, grad_accum_steps = new_batch_size, new_grad_accum_steps')
print("Wrote", "case_study_2/exp7/exp7_lora.py")

Wrote case_study_2/exp7/exp7_lora.py


In [15]:
(SRC_DIR / "case_study_2/exp7/exp7_nested_rank.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/exp7/exp7_nested_rank.py").write_text('from __future__ import annotations\n\nimport gc\nimport json\nimport time\nfrom dataclasses import dataclass, asdict\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import average_precision_score, precision_recall_curve, confusion_matrix\nimport matplotlib.pyplot as plt\n\nfrom case_study_2.data_loader import create_dataloader, get_class_weights\nfrom case_study_2.models import configure_huggingface_cache, load_code_tokenizer, DEFAULT_NEOBERT_TOKENIZER\nfrom case_study_2.exp7.exp7_lora import train_lora_model_safe, score_model\nfrom case_study_1 import split_manifest\nfrom case_study_1 import evaluation\nfrom case_study_1.evaluation import EvaluationConfig\nfrom case_study_1.confidence_intervals import bootstrap_metric_ci, format_ci_report\n\n\nEXP7_VERSION = "cs2-exp7-neobert-lora-v1"\n\n\n@dataclass(frozen=True)\nclass Exp7Config:\n    experiment_name: str = "cs2_exp7_neobert_lora"\n\n    code_column: str = "normalized_code"\n    source_id_column: str = "source_row_id"\n    label_column: str = "label"\n    project_column: str = "project"\n    fold_column: str = "fold"\n\n    hf_cache_dir: Optional[str] = None\n    max_length: int = 512\n    train_batch_size: int = 16\n    grad_accum_steps: int = 2\n\n    # --- FIX (undertraining diagnosis, see handoff notes) -------------------\n    # These used to be a direct copy of Exp4Config\'s budget (epochs=3,\n    # search_epochs=1), tuned for CodeBERTa (6 layers, 84M params). The\n    # canonical NeoBERT (28 layers, 224M trainable-relevant params) refit\n    # showed training loss still falling steadily at epoch 4/4 (1.14 -> 1.02\n    # -> 0.94 -> 0.83), i.e. it was stopped before convergence.\n    #\n    # `epochs` / `search_epochs` are now treated as CEILINGS, not fixed\n    # counts: exp7_lora.train_lora_model has validation-PR-AUC-based early\n    # stopping (see early_stopping/patience/min_epochs below) and returns the\n    # best checkpoint seen, so raising these ceilings costs nothing when the\n    # model converges earlier -- it just gives it room to actually get there\n    # when it doesn\'t.\n    epochs: int = 10          # was: 3 (refit / canonical retrain ceiling)\n    search_epochs: int = 4    # was: 1 (per-rank-candidate search ceiling)\n    early_stopping: bool = True\n    patience: int = 2         # stop if val PR-AUC hasn\'t improved in 2 epochs\n    min_epochs: int = 2       # never stop before this many epochs, even if\n                               # epoch 1 "improved" trivially over the -1.0 floor\n\n    rank_grid: Tuple[int, ...] = (8, 16)\n    inner_n_splits: int = 3\n    inner_random_state: int = 20260707\n    decision_threshold: float = 0.50\n\n    search_n_splits: int = 4\n\n    num_workers: int = 6\n    eval_batch_size: int = 64\n\n    n_splits: int = 5\n    random_state: int = 42\n    verbose: bool = True\n\n\ndef _checkpoint_paths(output_dir: Path, outer_fold_id: int) -> Dict[str, Path]:\n    root = output_dir / "checkpoints"\n    root.mkdir(parents=True, exist_ok=True)\n    prefix = f"outer_fold_{outer_fold_id}"\n    return {\n        "predictions": root / f"{prefix}_predictions.parquet",\n        "selected": root / f"{prefix}_selected_rank.json",\n        "training": root / f"{prefix}_outer_training.json",\n    }\n\n\ndef _search_checkpoint_path(output_dir: Path, outer_fold_id: int) -> Path:\n    root = output_dir / "checkpoints"\n    root.mkdir(parents=True, exist_ok=True)\n    return root / f"outer_fold_{outer_fold_id}_search_progress.json"\n\n\ndef _load_search_checkpoint(output_dir: Path, outer_fold_id: int) -> Dict[str, float]:\n    path = _search_checkpoint_path(output_dir, outer_fold_id)\n    if not path.exists():\n        return {}\n    with path.open("r", encoding="utf-8") as f:\n        raw = json.load(f)\n    return {int(k): float(v) for k, v in raw.items()}\n\n\ndef _save_search_checkpoint(output_dir: Path, outer_fold_id: int, rank_performance: Dict[int, float]) -> None:\n    path = _search_checkpoint_path(output_dir, outer_fold_id)\n    with path.open("w", encoding="utf-8") as f:\n        json.dump({str(k): v for k, v in rank_performance.items()}, f, indent=2)\n\n\ndef _write_outer_checkpoint(output_dir: Path, outer_fold_id: int, predictions: pd.DataFrame, selected: dict, training: dict) -> None:\n    paths = _checkpoint_paths(output_dir, outer_fold_id)\n    predictions.to_parquet(paths["predictions"], index=False)\n    with paths["selected"].open("w", encoding="utf-8") as f:\n        json.dump(selected, f, indent=2, default=str)\n    with paths["training"].open("w", encoding="utf-8") as f:\n        json.dump(training, f, indent=2, default=str)\n\n\ndef _load_outer_checkpoint(output_dir: Path, outer_fold_id: int) -> Optional[dict]:\n    paths = _checkpoint_paths(output_dir, outer_fold_id)\n    if not all(p.exists() for p in paths.values()):\n        return None\n    with paths["selected"].open("r", encoding="utf-8") as f:\n        selected = json.load(f)\n    with paths["training"].open("r", encoding="utf-8") as f:\n        training = json.load(f)\n    return {\n        "predictions": pd.read_parquet(paths["predictions"]),\n        "selected": selected,\n        "training": training,\n    }\n\n\ndef _update_run_state(state_path: Path, completed_folds, status: str) -> None:\n    state = {\n        "status": status,\n        "updated_utc": datetime.now(timezone.utc).isoformat(),\n        "completed_outer_folds": sorted(int(f) for f in completed_folds),\n    }\n    with state_path.open("w", encoding="utf-8") as f:\n        json.dump(state, f, indent=2)\n\n\ndef run_exp7_nested_rank(\n    development_frame: pd.DataFrame,\n    development_manifest: pd.DataFrame,\n    config: Exp7Config,\n    output_dir: Path,\n    resume: bool = True,\n    additional_metadata: Optional[Dict[str, Any]] = None,\n) -> Dict[str, Any]:\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    if device.type != "cuda":\n        raise RuntimeError("EXP-7 LoRA fine-tuning requires a CUDA device.")\n\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    state_path = output_dir / "exp7_nested_run_state.json"\n\n    configure_huggingface_cache(config.hf_cache_dir)\n    tokenizer = load_code_tokenizer(DEFAULT_NEOBERT_TOKENIZER, hf_cache_dir=config.hf_cache_dir)\n\n    fold_ids = sorted(development_manifest[config.fold_column].unique().tolist())\n    print(f"[nested] Starting EXP-7 nested rank search (NeoBERT LoRA) over {len(fold_ids)} outer folds, rank_grid={config.rank_grid}")\n    print(f"[nested] Search phase: {config.search_epochs} epoch(s), single held-out split | Refit phase: {config.epochs} epoch(s), full training")\n\n    oof_parts = []\n    selected_rows = []\n    outer_training_rows = []\n    completed_folds = []\n\n    t0 = time.time()\n\n    for outer_fold_id in fold_ids:\n        checkpoint = _load_outer_checkpoint(output_dir, outer_fold_id) if resume else None\n        if checkpoint is not None:\n            oof_parts.append(checkpoint["predictions"])\n            selected_rows.append(checkpoint["selected"])\n            outer_training_rows.append(checkpoint["training"])\n            completed_folds.append(outer_fold_id)\n            if config.verbose:\n                print(f"[nested] Outer fold {outer_fold_id}: loaded from checkpoint, skipping.")\n            continue\n\n        fold_t0 = time.time()\n        print(f"\\n=================== OUTER FOLD {outer_fold_id} ({len(completed_folds)+1}/{len(fold_ids)}) ===================")\n\n        outer_train_ids = development_manifest.loc[\n            development_manifest[config.fold_column] != outer_fold_id, config.source_id_column\n        ]\n        outer_val_ids = development_manifest.loc[\n            development_manifest[config.fold_column] == outer_fold_id, config.source_id_column\n        ]\n        outer_train_df = development_frame[development_frame[config.source_id_column].isin(outer_train_ids)].reset_index(drop=True)\n        outer_val_df = development_frame[development_frame[config.source_id_column].isin(outer_val_ids)].reset_index(drop=True)\n        print(f"[nested] outer_train={len(outer_train_df)} rows | outer_val={len(outer_val_df)} rows")\n\n        search_split_config = split_manifest.SplitConfig(\n            n_splits=config.search_n_splits,\n            random_state=config.inner_random_state,\n            shuffle=True,\n            source_id_column=config.source_id_column,\n            label_column=config.label_column,\n            group_column=config.project_column,\n        )\n        search_manifest = split_manifest.create_project_grouped_manifest(\n            outer_train_df[[config.source_id_column, config.label_column, config.project_column]],\n            config=search_split_config,\n        )\n        search_train_ids = search_manifest.loc[search_manifest["fold"] != 0, config.source_id_column]\n        search_val_ids = search_manifest.loc[search_manifest["fold"] == 0, config.source_id_column]\n        search_train_df = outer_train_df[outer_train_df[config.source_id_column].isin(search_train_ids)]\n        search_val_df = outer_train_df[outer_train_df[config.source_id_column].isin(search_val_ids)]\n        print(f"[nested] rank search split: train={len(search_train_df)} rows | val={len(search_val_df)} rows")\n\n        rank_performance = _load_search_checkpoint(output_dir, outer_fold_id) if resume else {}\n        if rank_performance:\n            print(f"[nested] resuming rank search, already have: {rank_performance}")\n\n        for rank_candidate in config.rank_grid:\n            if rank_candidate in rank_performance:\n                print(f"[nested] rank={rank_candidate}: already evaluated (PR-AUC={rank_performance[rank_candidate]:.4f}), skipping.")\n                continue\n\n            rank_t0 = time.time()\n            print(f"[nested] --- evaluating rank candidate {rank_candidate} ---")\n\n            val_scores, tmp_model = train_lora_model_safe(\n                search_train_df, search_val_df, tokenizer, rank=rank_candidate,\n                epochs=config.search_epochs, batch_size=config.train_batch_size,\n                grad_accum_steps=config.grad_accum_steps, eval_batch_size=config.eval_batch_size,\n                num_workers=config.num_workers, device=device,\n                hf_cache_dir=config.hf_cache_dir, code_column=config.code_column,\n                max_length=config.max_length, log_prefix="    ",\n                early_stopping=config.early_stopping, patience=config.patience,\n                min_epochs=config.min_epochs,\n            )\n            prauc = float(average_precision_score(search_val_df[config.label_column].values, val_scores))\n            rank_performance[rank_candidate] = prauc\n\n            print(f"[nested] rank={rank_candidate} | PR-AUC={prauc:.4f} | {(time.time()-rank_t0)/60:.1f} min")\n\n            _save_search_checkpoint(output_dir, outer_fold_id, rank_performance)\n\n            del tmp_model\n            gc.collect()\n            torch.cuda.empty_cache()\n\n        optimal_rank = max(rank_performance, key=rank_performance.get)\n        print(f"[nested] Selected rank={optimal_rank} for outer fold {outer_fold_id} | scores={rank_performance}")\n\n        print(f"[nested] --- final refit on full outer_train, rank={optimal_rank}, {config.epochs} epochs ---")\n        refit_t0 = time.time()\n        outer_val_scores, final_outer_model = train_lora_model_safe(\n            outer_train_df, outer_val_df, tokenizer, rank=optimal_rank,\n            epochs=config.epochs, batch_size=config.train_batch_size,\n            grad_accum_steps=config.grad_accum_steps, eval_batch_size=config.eval_batch_size,\n            num_workers=config.num_workers, device=device,\n            hf_cache_dir=config.hf_cache_dir, code_column=config.code_column,\n            max_length=config.max_length, log_prefix="    ",\n            early_stopping=config.early_stopping, patience=config.patience,\n            min_epochs=config.min_epochs,\n        )\n        print(f"[nested] refit done in {(time.time()-refit_t0)/60:.1f} min")\n\n        fold_oof = pd.DataFrame({\n            config.source_id_column: outer_val_df[config.source_id_column].values,\n            config.project_column: outer_val_df[config.project_column].values,\n            "label": outer_val_df[config.label_column].astype(int).values,\n            "y_score": outer_val_scores,\n            "fold": outer_fold_id,\n        })\n\n        selected_row = {"outer_fold_id": outer_fold_id, "selected_rank": optimal_rank, "search_scores": rank_performance}\n        training_row = {\n            "outer_fold_id": outer_fold_id,\n            "selected_rank": optimal_rank,\n            "n_train": int(len(outer_train_df)),\n            "n_val": int(len(outer_val_df)),\n            "elapsed_minutes": (time.time() - fold_t0) / 60,\n        }\n\n        if outer_fold_id == fold_ids[-1]:\n            final_outer_model.save_pretrained(output_dir / "final_exp7_lora_adapter")\n            print(f"[nested] saved final fold LoRA adapter to {output_dir / \'final_exp7_lora_adapter\'}")\n\n        _write_outer_checkpoint(output_dir, outer_fold_id, fold_oof, selected_row, training_row)\n\n        oof_parts.append(fold_oof)\n        selected_rows.append(selected_row)\n        outer_training_rows.append(training_row)\n        completed_folds.append(outer_fold_id)\n\n        _update_run_state(state_path, completed_folds, status="running")\n\n        del outer_train_df, outer_val_df, search_train_df, search_val_df, final_outer_model\n        gc.collect()\n        torch.cuda.empty_cache()\n\n        print(f"[nested] Outer fold {outer_fold_id} done in {training_row[\'elapsed_minutes\']:.1f} min | checkpoint saved | total elapsed {(time.time()-t0)/60:.1f} min")\n\n    oof_predictions = pd.concat(oof_parts, axis=0).reset_index(drop=True)\n\n    eval_config = EvaluationConfig(threshold=config.decision_threshold, expected_n_folds=len(fold_ids))\n    eval_results = evaluation.evaluate_oof_predictions(oof_predictions, config=eval_config)\n\n    selected_df = pd.DataFrame(selected_rows)\n    outer_training_df = pd.DataFrame(outer_training_rows)\n\n    artifacts = {\n        "oof_predictions": output_dir / "exp7_nested_oof_predictions.parquet",\n        "selected_rank_per_fold": output_dir / "exp7_selected_rank_per_outer_fold.csv",\n        "outer_training_audit": output_dir / "exp7_outer_training_audit.csv",\n        "run_metadata": output_dir / "exp7_nested_run_metadata.json",\n    }\n    oof_predictions.to_parquet(artifacts["oof_predictions"], index=False)\n    selected_df.to_csv(artifacts["selected_rank_per_fold"], index=False)\n    outer_training_df.to_csv(artifacts["outer_training_audit"], index=False)\n\n    metadata = {\n        "exp7_version": EXP7_VERSION,\n        "config": {**asdict(config), "rank_grid": list(config.rank_grid)},\n        "runtime_seconds": time.time() - t0,\n        **(additional_metadata or {}),\n    }\n    with open(artifacts["run_metadata"], "w", encoding="utf-8") as f:\n        json.dump(metadata, f, indent=2, default=str)\n\n    _update_run_state(state_path, completed_folds, status="completed")\n\n    print(f"\\n[nested] EXP-7 nested rank search complete in {(time.time()-t0)/60:.1f} min")\n\n    return {\n        "oof_predictions": oof_predictions,\n        "evaluation": eval_results,\n        "selected_rank": selected_df,\n        "outer_fold_training": outer_training_df,\n        "artifacts": artifacts,\n        "tokenizer": tokenizer,\n    }\n\n\ndef run_exp7_canonical_retrain(\n    development_frame: pd.DataFrame,\n    tokenizer,\n    selected_rank: int,\n    holdout_frame: pd.DataFrame,\n    config: Exp7Config,\n    output_dir: Path,\n) -> Dict[str, Any]:\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    # --- FIX (leakage guard) -------------------------------------------\n    # train_lora_model now does early stopping against whatever `val_df`\n    # it is given, and keeps the checkpoint with the best val PR-AUC. The\n    # OLD code passed `holdout_frame` straight in as that val_df, which was\n    # harmless when val_df was only used for post-hoc scoring (fixed epoch\n    # count), but would become a genuine leak now: early stopping would be\n    # selecting the "best" epoch using the frozen outer holdout, which is\n    # supposed to be touched exactly once, at final evaluation time.\n    #\n    # Fix: carve a small internal validation split out of development_frame\n    # (project-grouped, like the rank-search split above) purely for early\n    # stopping, train against THAT, and only afterwards run one no-gradient\n    # scoring pass over holdout_frame with the selected best checkpoint.\n    canonical_val_config = split_manifest.SplitConfig(\n        n_splits=config.search_n_splits,\n        random_state=config.inner_random_state,\n        shuffle=True,\n        source_id_column=config.source_id_column,\n        label_column=config.label_column,\n        group_column=config.project_column,\n    )\n    canonical_manifest = split_manifest.create_project_grouped_manifest(\n        development_frame[[config.source_id_column, config.label_column, config.project_column]],\n        config=canonical_val_config,\n    )\n    canonical_train_ids = canonical_manifest.loc[canonical_manifest["fold"] != 0, config.source_id_column]\n    canonical_val_ids = canonical_manifest.loc[canonical_manifest["fold"] == 0, config.source_id_column]\n    canonical_train_df = development_frame[development_frame[config.source_id_column].isin(canonical_train_ids)].reset_index(drop=True)\n    canonical_val_df = development_frame[development_frame[config.source_id_column].isin(canonical_val_ids)].reset_index(drop=True)\n\n    # --- CHECKPOINT (canonical retrain) --------------------------------\n    # Canonical retrain has no per-epoch checkpointing, so a crash AFTER\n    # save_pretrained() but before/during holdout scoring (e.g. the\n    # score_model NameError) used to mean re-running the full ~2hr training\n    # from scratch just to redo a scoring pass. If a saved adapter already\n    # exists at this path, reload it and skip straight to scoring instead.\n    canonical_model_path = output_dir / "final_canonical_lora_model"\n    if canonical_model_path.exists():\n        from peft import PeftModel\n        from case_study_2.models import CodeSequenceClassifier, DEFAULT_NEOBERT_MODEL\n\n        print(f"[canonical] Found existing model at {canonical_model_path}, skipping training and loading it.")\n        base_model = CodeSequenceClassifier(\n            model_name=DEFAULT_NEOBERT_MODEL, freeze_backbone=False,\n            pooling="mean", hf_cache_dir=config.hf_cache_dir,\n        )\n        global_model = PeftModel.from_pretrained(base_model, str(canonical_model_path)).to(device)\n    else:\n        print(\n            f"[canonical] Retraining with internal early-stopping split: "\n            f"train={len(canonical_train_df)} rows | internal_val={len(canonical_val_df)} rows "\n            f"(holdout, {len(holdout_frame)} rows, is NOT used for early stopping) | "\n            f"rank={selected_rank}, up to {config.epochs} epochs"\n        )\n        t0 = time.time()\n\n        _, global_model = train_lora_model_safe(\n            canonical_train_df, canonical_val_df, tokenizer, rank=selected_rank,\n            epochs=config.epochs, batch_size=config.train_batch_size,\n            grad_accum_steps=config.grad_accum_steps, eval_batch_size=config.eval_batch_size,\n            num_workers=config.num_workers, device=device,\n            hf_cache_dir=config.hf_cache_dir, code_column=config.code_column,\n            max_length=config.max_length, log_prefix="  ",\n            early_stopping=config.early_stopping, patience=config.patience,\n            min_epochs=config.min_epochs,\n        )\n        global_model.save_pretrained(canonical_model_path)\n        print(f"[canonical] Training done in {(time.time()-t0)/60:.1f} min | model saved to {canonical_model_path}")\n\n    # Single, no-gradient scoring pass over the frozen outer holdout with the\n    # selected (best-val-PR-AUC) checkpoint. This is the ONLY point in the\n    # whole EXP-7 pipeline where the model touches holdout_frame.\n    print(f"[canonical] Scoring frozen outer holdout ({len(holdout_frame)} rows)...")\n    holdout_scores = score_model(\n        global_model, holdout_frame, tokenizer, device,\n        code_column=config.code_column, max_length=config.max_length,\n        batch_size=config.eval_batch_size, num_workers=config.num_workers,\n    )\n\n    holdout_predictions = pd.DataFrame({\n        config.source_id_column: holdout_frame[config.source_id_column].values,\n        config.project_column: holdout_frame[config.project_column].values,\n        "label": holdout_frame[config.label_column].astype(int).values,\n        "y_score": holdout_scores,\n        "fold": 0,\n    })\n\n    del global_model\n    gc.collect()\n    torch.cuda.empty_cache()\n\n    return {"holdout_predictions": holdout_predictions}\n\n\ndef run_exp7_holdout_evaluation(\n    holdout_predictions: pd.DataFrame,\n    config: Exp7Config,\n    output_dir: Path,\n    n_bootstrap: int = 1000,\n    confidence: float = 0.95,\n) -> Dict[str, Any]:\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    holdout_predictions.to_csv(output_dir / "exp7_holdout_predictions.csv", index=False)\n\n    eval_config = EvaluationConfig(threshold=config.decision_threshold, expected_n_folds=1)\n    holdout_metrics = evaluation.evaluate_oof_predictions(holdout_predictions, config=eval_config)\n    print(evaluation.format_metric_report(holdout_metrics["pooled_metrics"]))\n\n    ci_result = bootstrap_metric_ci(\n        holdout_predictions,\n        metric="average_precision_pr_auc",\n        group_column="project",\n        n_bootstrap=n_bootstrap,\n        confidence=confidence,\n        random_state=config.random_state,\n    )\n    with open(output_dir / "exp7_holdout_pr_auc_bootstrap_ci.json", "w", encoding="utf-8") as f:\n        json.dump(ci_result.as_dict(), f, indent=2)\n    print(format_ci_report(ci_result))\n\n    y_true = holdout_predictions["label"].values\n    y_score = holdout_predictions["y_score"].values\n    precision, recall, _ = precision_recall_curve(y_true, y_score)\n    ap = holdout_metrics["pooled_metrics"]["average_precision_pr_auc"]\n    plt.figure(figsize=(6, 5))\n    plt.plot(recall, precision, color="b", label=f"EXP-7 LoRA/NeoBERT (PR-AUC = {ap:.4f})")\n    plt.xlabel("Recall")\n    plt.ylabel("Precision")\n    plt.title("Precision-Recall Curve - Frozen Outer Holdout")\n    plt.legend(loc="lower left")\n    plt.grid(True)\n    plt.savefig(output_dir / "exp7_outer_holdout_pr_curve.png")\n    plt.close()\n\n    y_pred = (y_score >= config.decision_threshold).astype(int)\n    cm = confusion_matrix(y_true, y_pred)\n    plt.figure(figsize=(4, 4))\n    plt.imshow(cm, cmap=plt.cm.Blues)\n    plt.title("Confusion Matrix - Frozen Outer Holdout")\n    plt.xlabel("Predicted")\n    plt.ylabel("Actual")\n    plt.xticks([0, 1], ["Non-Vuln (0)", "Vuln (1)"])\n    plt.yticks([0, 1], ["Non-Vuln (0)", "Vuln (1)"])\n    for i in range(2):\n        for j in range(2):\n            plt.text(j, i, str(cm[i, j]), ha="center", va="center")\n    plt.tight_layout()\n    plt.savefig(output_dir / "exp7_outer_holdout_confusion_matrix.png")\n    plt.close()\n\n    return {\n        "holdout_metrics": holdout_metrics,\n        "bootstrap_ci": ci_result.as_dict(),\n        "y_pred": y_pred,\n    }')
print("Wrote", "case_study_2/exp7/exp7_nested_rank.py")

Wrote case_study_2/exp7/exp7_nested_rank.py


## 6. Import project modules


In [16]:
import sys

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("case_study_2") or mod_name.startswith("case_study_1"):
        del sys.modules[mod_name]

from case_study_2.data_loader import create_dataloader, get_class_weights
from case_study_2.models import (
    configure_huggingface_cache,
    load_code_tokenizer,
    DEFAULT_NEOBERT_MODEL,
    DEFAULT_NEOBERT_TOKENIZER,
    count_trainable_parameters,
    CodeSequenceClassifier,
    infer_lora_target_modules,
)
from case_study_2.exp7.exp7_lora import train_lora_model, train_lora_model_safe
from case_study_2.exp7.exp7_nested_rank import (
    Exp7Config,
    run_exp7_nested_rank,
    run_exp7_canonical_retrain,
    run_exp7_holdout_evaluation,
)
from case_study_1.confidence_intervals import paired_bootstrap_metric_ci, format_paired_ci_report


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 7. Load dataset and frozen manifests


In [17]:
import pandas as pd

full_df = pd.read_parquet(NORMALIZED_PARQUET)
outer_manifest_df = pd.read_parquet(OUTER_MANIFEST_PATH)
inner_manifest_df = pd.read_parquet(INNER_MANIFEST_PATH)

print("full_df rows:", len(full_df))
print("outer_manifest_df rows:", len(outer_manifest_df))
print("inner_manifest_df rows:", len(inner_manifest_df))


full_df rows: 261667
outer_manifest_df rows: 261667
inner_manifest_df rows: 203958


## 8. Build development and holdout frames


In [18]:
required_columns = {"source_row_id", "normalized_code", "label", "project"}
missing_columns = required_columns - set(full_df.columns)
if missing_columns:
    raise ValueError(f"Missing columns in full_df: {missing_columns}")

full_indexed = full_df.set_index("source_row_id", drop=False)
dev_ids = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "development", "source_row_id"].tolist())
holdout_ids = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "outer_holdout", "source_row_id"].tolist())
inner_ids = set(inner_manifest_df["source_row_id"].tolist())

if dev_ids != inner_ids:
    raise RuntimeError("Development partition and inner manifest coverage do not match.")
if dev_ids.intersection(holdout_ids):
    raise RuntimeError("Development and holdout partitions overlap.")

development_frame = full_indexed.loc[full_indexed["source_row_id"].isin(dev_ids)].reset_index(drop=True)
holdout_frame = full_indexed.loc[full_indexed["source_row_id"].isin(holdout_ids)].reset_index(drop=True)

print("development_frame rows:", len(development_frame))
print("holdout_frame rows:", len(holdout_frame))


development_frame rows: 203958
holdout_frame rows: 57709


## 9. Configuration object


In [ ]:
exp7_config = Exp7Config(
    hf_cache_dir=HF_CACHE_DIR,
    code_column=CODE_COLUMN,
    rank_grid=RANK_GRID,
    epochs=EPOCHS,
    search_epochs=SEARCH_EPOCHS,
    search_n_splits=SEARCH_N_SPLITS,
    train_batch_size=TRAIN_BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    num_workers=NUM_WORKERS,
    eval_batch_size=EVAL_BATCH_SIZE,
)


## 10. Sanity-check LoRA target modules


In [20]:
configure_huggingface_cache(HF_CACHE_DIR)
_tok_check = load_code_tokenizer(DEFAULT_NEOBERT_TOKENIZER, hf_cache_dir=HF_CACHE_DIR)

_probe_model = CodeSequenceClassifier(model_name=DEFAULT_NEOBERT_MODEL, freeze_backbone=False, dtype_policy="bfloat16", hf_cache_dir=HF_CACHE_DIR)
_target_modules = infer_lora_target_modules(_probe_model)
print("Inferred LoRA target_modules:", _target_modules)

del _probe_model
torch.cuda.empty_cache()


2026-08-10 16:59:35.894682: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-10 16:59:35.909256: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786381175.925249   22728 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786381175.930107   22728 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-10 16:59:35.947792: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Inferred LoRA target_modules: ['qkv']


## 11. Smoke test on a small subsample


In [21]:
import time
from sklearn.metrics import average_precision_score

if RUN_SMOKE_TEST:
    sample_df = development_frame.sample(n=min(2000, len(development_frame)), random_state=42).reset_index(drop=True)
    smoke_train = sample_df.iloc[:1500].reset_index(drop=True)
    smoke_val = sample_df.iloc[1500:].reset_index(drop=True)

    print(f"[smoke] train={len(smoke_train)} rows | val={len(smoke_val)} rows")

    t0 = time.time()
    smoke_scores, smoke_model = train_lora_model_safe(
        smoke_train, smoke_val, _tok_check, rank=RANK_GRID[0], epochs=1,
        batch_size=exp7_config.train_batch_size, grad_accum_steps=exp7_config.grad_accum_steps,
        eval_batch_size=exp7_config.eval_batch_size, num_workers=exp7_config.num_workers,
        device=DEVICE, hf_cache_dir=HF_CACHE_DIR, code_column=exp7_config.code_column,
        max_length=exp7_config.max_length,
    )
    smoke_prauc = float(average_precision_score(smoke_val[exp7_config.label_column].values, smoke_scores))
    print(f"[smoke] PR-AUC={smoke_prauc:.4f} | elapsed={(time.time()-t0)/60:.1f} min")

    del smoke_model
    torch.cuda.empty_cache()
else:
    print("RUN_SMOKE_TEST=False; skipping.")


[smoke] train=1500 rows | val=500 rows


/workspace/DiverseVul--IS-Project/vuln-detection/src/case_study_2/models.py:91: UserWarning: [models] Could not find a known unpadding flag on the NeoBERT config (checked: ('use_unpadding', 'unpad_inputs', 'unpad', 'pack_sequences')). This revision of chandar-lab/NeoBERT may handle padding differently -- double check attention-mask correctness manually (see PDD sec. 5.2 / GitHub Issue #7).
  warnings.warn(


[lora] rank=8 | train_rows=1500 | val_rows=500 | batch_size=32 | grad_accum=2 | steps/epoch=47 | trainable=688,897 (0.310%) | total=222,355,202
[lora] epoch 1/1 done | avg_loss=1.4768 | val_pr_auc=0.1066 <- best so far | epoch_time=0.3 min | total_elapsed=0.3 min | peak_VRAM=0.00 GB
[lora] training done (1 epoch(s) run).
[lora] restored best checkpoint from epoch 1 (val_pr_auc=0.1066).
[smoke] PR-AUC=0.1066 | elapsed=0.4 min


## 12. Official nested LoRA rank search


In [22]:
if RUN_NESTED_OFFICIAL:
    nested_results = run_exp7_nested_rank(
        development_frame=development_frame,
        development_manifest=inner_manifest_df,
        config=exp7_config,
        output_dir=EXP7_OUTPUT_DIR,
        additional_metadata={
            "input_parquet": str(NORMALIZED_PARQUET),
            "outer_manifest_path": str(OUTER_MANIFEST_PATH),
            "inner_manifest_path": str(INNER_MANIFEST_PATH),
        },
    )
    display(nested_results["selected_rank"])
    print("Pooled nested PR-AUC:", nested_results["evaluation"]["pooled_metrics"]["average_precision_pr_auc"])
    tokenizer = nested_results["tokenizer"]
else:
    nested_results = None
    tokenizer = _tok_check
    print("RUN_NESTED_OFFICIAL=False; skipping.")


[nested] Starting EXP-7 nested rank search (NeoBERT LoRA) over 5 outer folds, rank_grid=(8, 16, 32)
[nested] Search phase: 2 epoch(s), single held-out split | Refit phase: 4 epoch(s), full training
[nested] Outer fold 0: loaded from checkpoint, skipping.
[nested] Outer fold 1: loaded from checkpoint, skipping.
[nested] Outer fold 2: loaded from checkpoint, skipping.
[nested] Outer fold 3: loaded from checkpoint, skipping.
[nested] Outer fold 4: loaded from checkpoint, skipping.

[nested] EXP-7 nested rank search complete in 0.2 min


,outer_fold_id,selected_rank,search_scores
0,0,32,"{'8': 0.1336324904071973, '16': 0.127308611595..."
1,1,8,"{'8': 0.5476190476190477, '16': 0.123809523809..."
2,2,32,"{'8': 0.0701209881807892, '16': 0.061178632078..."
3,3,16,"{'8': 0.33009755291005294, '16': 0.56287210338..."
4,4,32,"{'8': 0.42857142857142855, '16': 0.30882352941..."


Pooled nested PR-AUC: 0.11396129091702312


## 13. Canonical retrain and frozen outer holdout scoring


In [23]:
if RUN_CANONICAL_RETRAIN and nested_results is not None:
    global_selected_rank = int(nested_results["selected_rank"]["selected_rank"].mode()[0])
    retrain_results = run_exp7_canonical_retrain(
        development_frame=development_frame,
        tokenizer=tokenizer,
        selected_rank=global_selected_rank,
        holdout_frame=holdout_frame,
        config=exp7_config,
        output_dir=EXP7_OUTPUT_DIR,
    )
    print("Canonical model trained with rank =", global_selected_rank)
else:
    retrain_results = None
    print("RUN_CANONICAL_RETRAIN=False or no nested results; skipping.")


[canonical] Found existing model at /workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs/case_study_2/exp7_neobert_lora_v1/final_canonical_lora_model, skipping training and loading it.


/workspace/DiverseVul--IS-Project/vuln-detection/src/case_study_2/models.py:91: UserWarning: [models] Could not find a known unpadding flag on the NeoBERT config (checked: ('use_unpadding', 'unpad_inputs', 'unpad', 'pack_sequences')). This revision of chandar-lab/NeoBERT may handle padding differently -- double check attention-mask correctness manually (see PDD sec. 5.2 / GitHub Issue #7).
  warnings.warn(


[canonical] Scoring frozen outer holdout (57709 rows)...
Canonical model trained with rank = 32


## 14. Frozen outer holdout evaluation with bootstrap confidence interval


In [25]:
if RUN_HOLDOUT_EVAL and retrain_results is not None:
    holdout_results = run_exp7_holdout_evaluation(
        holdout_predictions=retrain_results["holdout_predictions"],
        config=exp7_config,
        output_dir=EXP7_OUTPUT_DIR,
    )
else:
    holdout_results = None
    print("RUN_HOLDOUT_EVAL=False or no canonical retrain results; skipping.")


Pooled Out-of-Fold Evaluation
                   n_samples: 57709
                vulnerable_1: 3211
            non_vulnerable_0: 54498
               positive_rate: 0.055641
                   threshold: 0.500000
    average_precision_pr_auc: 0.125917
                   precision: 0.111603
                      recall: 0.510744
                          f1: 0.183179
                         mcc: 0.142693
                 specificity: 0.760450
         false_positive_rate: 0.239550
               true_negative: 41443
              false_positive: 13055
              false_negative: 1571
               true_positive: 1640
average_precision_pr_auc: point estimate = 0.1259
  95% CI (project-block bootstrap): [0.1066, 0.1520]
  valid resamples: 1000/1000 (0 degenerate, dropped)
  n_projects: 203, random_state=42
  Reflects sampling variability within this dataset only; not an estimate of generalization to C functions outside this collection.


## 15. Paired comparison against EXP-4 on the frozen holdout


In [26]:
EXP4_HOLDOUT_PREDICTIONS_PATH = OUTPUT_ROOT / "case_study_2" / "exp4_codeberta_lora_v1" / "exp4_holdout_predictions.csv"

if RUN_HOLDOUT_EVAL and retrain_results is not None and EXP4_HOLDOUT_PREDICTIONS_PATH.exists():
    exp4_holdout = pd.read_csv(EXP4_HOLDOUT_PREDICTIONS_PATH)
    comparison = paired_bootstrap_metric_ci(
        predictions_a=retrain_results["holdout_predictions"],
        predictions_b=exp4_holdout,
        experiment_name_a="EXP-7 LoRA (NeoBERT)",
        experiment_name_b="EXP-4 LoRA (CodeBERTa)",
        metric="average_precision_pr_auc",
    )
    print(format_paired_ci_report(comparison))
else:
    print("Run EXP-7 holdout evaluation and the EXP-4 notebook first.")


average_precision_pr_auc: EXP-7 LoRA (NeoBERT) = 0.1259, EXP-4 LoRA (CodeBERTa) = 0.1439
  Difference (EXP-7 LoRA (NeoBERT) - EXP-4 LoRA (CodeBERTa)) = -0.0179
  95% CI on the difference (paired project-block bootstrap): [-0.0342, -0.0043]
  valid resamples: 1000/1000 (0 degenerate, dropped)
  An interval excluding zero means the difference is unlikely to be bootstrap noise within this dataset; it says nothing about generalization beyond it.


## 16. Cleanup


In [27]:
import gc
import shutil
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("VRAM allocated:", torch.cuda.memory_allocated() / 1e9, "GB")

total, used, free = shutil.disk_usage(WORKSPACE_ROOT)
print(f"Disk usage at {WORKSPACE_ROOT}: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")
if used / 1e9 > STORAGE_CAP_GB:
    print(f"WARNING: workspace usage exceeds the {STORAGE_CAP_GB} GB storage cap -- consider pruning old checkpoints under {EXP7_OUTPUT_DIR}.")


VRAM allocated: 0.01703936 GB
Disk usage at /workspace: 5223.2 GB used / 5714.2 GB total (202.9 GB free)
